# MemoryBandit Trial-wise Multivariate Imaginary Coherency

Compute trial-wise multivariate maximised imaginary coherency (MIC) between `HPC` and each target ROI during feedback and cue epochs.

The exported table contains only shuffle-z-scored MIC values (`mic_z`) plus trial/behavior metadata, so RPE-category or other posthoc splits can be done without recomputing connectivity.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import json
import hashlib
import warnings
from copy import deepcopy

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

from plot_config import set_plot_style
from mv_granger_utils import (
    load_behavior_tables,
    preprocess_learn_df,
    preprocess_combined_df,
    load_subject_epochs,
)

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

try:
    from mne_connectivity import spectral_connectivity_time
except Exception:
    spectral_connectivity_time = None

warnings.filterwarnings('ignore')
set_plot_style()
np.set_printoptions(suppress=True, precision=4)


In [ ]:
# Paths and subject list
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mvmic'
os.makedirs(savepath, exist_ok=True)

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
subjects = sorted(subj_df['MSSMCode'].dropna().unique().tolist())

learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

print('Subjects:', len(subjects))
print('combined_df:', combined_df.shape)
print('combined_df has RPE metadata:', {'participant', 'rpe', 'rpe_cat'}.issubset(combined_df.columns))


In [ ]:
# Analysis configuration
MIC_CFG = {
    'roi_source': 'HPC',
    'target_rois': ['ACC', 'AMY', 'dlPFC', 'dmPFC', 'OFC'],
    'events': {
        'feedback_start': {'tmin': 0.0, 'tmax': 1.5},
        'cue_on': {'tmin': 0.0, 'tmax': 1.0},
    },
    'day': 'Day1',
    'target_sfreq': 50,
    'fmin': 4,
    'fmax': 8,
    'n_freqs': 8,
    'n_cycles': 3,
    'random_state': 42,
    'min_shift': 2,
}

RUN = {
    'run_single_subject_smoke': True,
    'run_batch': False,
    'overwrite_subject_cache': False,
    'smoke_subject': 'UI002' if 'UI002' in subjects else subjects[0],
    'smoke_target_roi': 'ACC',
    'n_surrogates_smoke': 25,
    'n_surrogates_batch': 100,
    'n_jobs_batch': 6,
    'parallel_backend': 'loky',
}

ANALYSIS_STATE = globals().get('ANALYSIS_STATE', {})

def _analysis_value_token(value):
    return str(value).replace('.', 'p').replace('-', 'm')

def _stable_json_dumps(payload):
    return json.dumps(payload, sort_keys=True, separators=(',', ':'), default=str)

def _build_analysis_tag(cfg):
    key = {
        'roi_source': cfg['roi_source'],
        'target_rois': cfg['target_rois'],
        'events': cfg['events'],
        'day': cfg['day'],
        'target_sfreq': cfg['target_sfreq'],
        'fmin': cfg['fmin'],
        'fmax': cfg['fmax'],
        'n_freqs': cfg['n_freqs'],
        'n_cycles': cfg['n_cycles'],
    }
    cfg_hash = hashlib.sha1(_stable_json_dumps(key).encode('utf-8')).hexdigest()[:10]
    return (
        f"trialwise_{cfg['roi_source']}toTargets_{cfg['day']}"
        f"_sf{_analysis_value_token(cfg['target_sfreq'])}"
        f"_{cfg['fmin']}-{cfg['fmax']}Hz"
        f"_roi{cfg_hash}"
    )

analysis_tag = _build_analysis_tag(MIC_CFG)
analysis_dir = os.path.join(savepath, 'notebook_runs', analysis_tag)
subject_cache_dir = os.path.join(analysis_dir, 'subject_cache')
os.makedirs(subject_cache_dir, exist_ok=True)

analysis_paths = {
    'analysis_dir': analysis_dir,
    'subject_cache_dir': subject_cache_dir,
    'combined_csv': os.path.join(analysis_dir, 'trialwise_mvmic_z.csv'),
    'cache_log_csv': os.path.join(analysis_dir, 'cache_log.csv'),
    'summary_csv': os.path.join(analysis_dir, 'trialwise_mvmic_subject_summary.csv'),
    'bar_strip_pdf': os.path.join(analysis_dir, 'trialwise_mvmic_bar_strip.pdf'),
    'bar_strip_png': os.path.join(analysis_dir, 'trialwise_mvmic_bar_strip.png'),
    'rpe_split_pdf': os.path.join(analysis_dir, 'trialwise_mvmic_rpe_split_bar_strip.pdf'),
    'rpe_split_png': os.path.join(analysis_dir, 'trialwise_mvmic_rpe_split_bar_strip.png'),
}

print('Smoke subject:', RUN['smoke_subject'])
print('Smoke target:', MIC_CFG['roi_source'], '->', RUN['smoke_target_roi'])
print('Analysis dir:', analysis_dir)
print('Batch enabled:', RUN['run_batch'])


## Helper Functions

These helpers keep the notebook self-contained. The null distribution is built by circularly shifting only the source/HPC channels inside each trial, then recomputing trial-wise MIC.


In [ ]:
def resolve_trial_col_for_event(df, event):
    event_name = str(event).strip().lower()
    if event_name == 'cue_on':
        candidates = ('trials_mem', 'trials_gamble', 'trials')
    else:
        candidates = ('trials_gamble', 'trials', 'trials_mem')
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f'Could not resolve trial column for event={event!r}')

def normalize_rpe_cat(s):
    mapping = {
        'negative': 'neg',
        'neg': 'neg',
        'neutral': 'neutral',
        'positive': 'pos',
        'pos': 'pos',
    }
    return s.astype(str).str.strip().str.lower().map(mapping)

def build_trial_map(combined_df, subject, event, n_epochs, day='Day1'):
    trial_col = resolve_trial_col_for_event(combined_df, event)
    trial_map = combined_df[combined_df['participant'].eq(subject)].copy()
    if 'condition' in trial_map.columns:
        trial_map = trial_map[trial_map['condition'].eq(day)].copy()
    trial_map = trial_map.dropna(subset=[trial_col]).copy()
    if len(trial_map) == 0:
        return pd.DataFrame(), trial_col

    trial_map['trial_index'] = pd.to_numeric(trial_map[trial_col], errors='coerce').astype('Int64') - 1
    trial_map = trial_map.dropna(subset=['trial_index']).copy()
    trial_map['trial_index'] = trial_map['trial_index'].astype(int)
    trial_map = trial_map[(trial_map['trial_index'] >= 0) & (trial_map['trial_index'] < int(n_epochs))].copy()
    trial_map = trial_map.sort_values('trial_index').drop_duplicates(subset=['trial_index'], keep='first').reset_index(drop=True)

    if 'rpe_cat' in trial_map.columns:
        trial_map['rpe_cat'] = normalize_rpe_cat(trial_map['rpe_cat'])

    for col in ['trials_mem', 'trials_gamble', 'trials', 'rpe', 'rpe_cat', 'memory_outcome']:
        if col not in trial_map.columns:
            trial_map[col] = np.nan

    keep = ['participant', 'condition', 'trial_index', 'trials_mem', 'trials_gamble', 'trials', 'rpe', 'rpe_cat', 'memory_outcome']
    return trial_map[[c for c in keep if c in trial_map.columns]].reset_index(drop=True), trial_col

def make_multivariate_indices(source_idx, target_idx):
    source_idx = np.unique(np.asarray(source_idx, dtype=int))
    target_idx = np.unique(np.asarray(target_idx, dtype=int))
    return (np.array([source_idx.tolist()], dtype=object), np.array([target_idx.tolist()], dtype=object))

def extract_trial_mic_vector(con_obj, n_epochs, label):
    raw = np.asarray(con_obj.get_data())
    arr = np.squeeze(raw)
    arr = np.abs(arr.astype(complex, copy=False))

    if arr.ndim == 0:
        raise RuntimeError(f'{label}: expected one value per trial, got scalar')

    matching_axes = [axis for axis, size in enumerate(arr.shape) if size == int(n_epochs)]
    if len(matching_axes) == 0:
        raise RuntimeError(f'{label}: expected an axis with n_epochs={n_epochs}, got shape {arr.shape}')

    # Prefer the first matching axis, then average any remaining connection/frequency/time dimensions.
    arr = np.moveaxis(arr, matching_axes[0], 0)
    if arr.shape[0] != int(n_epochs):
        raise RuntimeError(f'{label}: trial axis move failed, got shape {arr.shape}')
    return np.nanmean(arr.reshape(int(n_epochs), -1), axis=1).astype(float)

def scalar_null_stats(real_value, surrogate_values, eps=1e-8):
    surrogate_values = np.asarray(surrogate_values, dtype=float)
    surrogate_values = surrogate_values[np.isfinite(surrogate_values)]
    if surrogate_values.size == 0 or not np.isfinite(real_value):
        return dict(z=np.nan, n=0)
    mean_val = float(np.nanmean(surrogate_values))
    std_val = float(max(np.nanstd(surrogate_values), eps))
    return dict(z=float((real_value - mean_val) / std_val), n=int(surrogate_values.size))

def stable_int_seed(*parts, modulo=1_000_000):
    token = _stable_json_dumps({'parts': [str(p) for p in parts]})
    return int(hashlib.sha1(token.encode('utf-8')).hexdigest()[:12], 16) % int(modulo)

def effective_n_cycles(epochs, cfg, safety=0.55):
    # MNE's Morlet wavelet spans roughly 10 sigma, so low-frequency wavelets
    # can exceed short cue epochs. Keep the requested value when safe, otherwise
    # reduce cycles enough for the shortest frequency to fit inside the epoch.
    duration = float(epochs.times[-1] - epochs.times[0]) if len(epochs.times) > 1 else 0.0
    if duration <= 0:
        return max(1.0, min(float(cfg['n_cycles']), 2.0))
    max_cycles = safety * duration * float(cfg['fmin'])
    return float(max(1.0, min(float(cfg['n_cycles']), max_cycles)))

def compute_trialwise_mic(epochs, indices, freqs, cfg, label='mic'):
    n_cycles_eff = effective_n_cycles(epochs, cfg)
    con = spectral_connectivity_time(
        epochs,
        sfreq=float(epochs.info['sfreq']),
        method=['mic'],
        indices=indices,
        fmin=cfg['fmin'],
        fmax=cfg['fmax'],
        freqs=freqs,
        n_cycles=n_cycles_eff,
        average=False,
        faverage=True,
        verbose='ERROR',
    )
    return extract_trial_mic_vector(con, len(epochs), label)

def shifted_source_epochs(epochs, source_idx, rng, min_shift=2):
    base_data = epochs.get_data(copy=True)
    source_idx = np.unique(np.asarray(source_idx, dtype=int))
    n_trials, _, n_times = base_data.shape
    candidate_shifts = np.arange(int(min_shift), n_times - int(min_shift))
    if candidate_shifts.size == 0:
        candidate_shifts = np.arange(1, n_times)
    if candidate_shifts.size == 0:
        raise RuntimeError('Cannot circular-shift source data with fewer than two timepoints')

    for trial_idx in range(n_trials):
        shift = int(rng.choice(candidate_shifts))
        base_data[trial_idx, source_idx, :] = np.roll(base_data[trial_idx, source_idx, :], shift=shift, axis=-1)

    return mne.EpochsArray(
        base_data,
        info=deepcopy(epochs.info),
        events=epochs.events.copy(),
        event_id=epochs.event_id.copy() if epochs.event_id is not None else None,
        tmin=float(epochs.tmin),
        baseline=epochs.baseline,
        verbose=False,
    )


In [ ]:
def compute_subject_event_target_mic(subject, event, target_roi, n_surrogates, verbose=False):
    epoch_window = MIC_CFG['events'][event]
    epoch_data = load_subject_epochs(
        subj_id=subject,
        base_dir=base_dir,
        event=event,
        roi_source=MIC_CFG['roi_source'],
        roi_target=target_roi,
        tmin=epoch_window['tmin'],
        tmax=epoch_window['tmax'],
        target_sfreq=MIC_CFG['target_sfreq'],
        day=MIC_CFG['day'],
        fmin=MIC_CFG['fmin'],
        fmax=MIC_CFG['fmax'],
    )
    if epoch_data is None:
        return pd.DataFrame(), {'subject': subject, 'event': event, 'roi_target': target_roi, 'skip_reason': 'epoch_load_failed'}

    epochs = epoch_data['epochs']
    trial_map, trial_col = build_trial_map(combined_df, subject, event, len(epochs), day=MIC_CFG['day'])
    if len(trial_map) == 0:
        return pd.DataFrame(), {'subject': subject, 'event': event, 'roi_target': target_roi, 'skip_reason': 'no_aligned_trials'}

    freqs = np.linspace(MIC_CFG['fmin'], MIC_CFG['fmax'], int(MIC_CFG['n_freqs']))
    rng = np.random.default_rng(
        MIC_CFG['random_state'] + stable_int_seed(subject, event, target_roi)
    )

    real_by_hemi = {}
    surr_by_hemi = {}
    for hemi in ('l', 'r'):
        source_idx = np.unique(np.asarray(epoch_data.get(f'source_indices_{hemi}', []), dtype=int))
        target_idx = np.unique(np.asarray(epoch_data.get(f'target_indices_{hemi}', []), dtype=int))
        if source_idx.size == 0 or target_idx.size == 0:
            continue

        indices = make_multivariate_indices(source_idx, target_idx)
        try:
            real = compute_trialwise_mic(epochs, indices, freqs, MIC_CFG, label=f'{subject} {event} {target_roi} {hemi} real')
        except Exception as exc:
            if verbose:
                print(f'{subject} {event} {target_roi} {hemi}: real MIC failed:', exc)
            continue

        surr = np.full((int(n_surrogates), len(epochs)), np.nan, dtype=float)
        iterator = range(int(n_surrogates))
        if verbose:
            iterator = tqdm(iterator, desc=f'{subject} {event} {target_roi} {hemi} surrogates')
        for s_idx in iterator:
            try:
                shifted_epochs = shifted_source_epochs(
                    epochs,
                    source_idx=source_idx,
                    rng=rng,
                    min_shift=MIC_CFG['min_shift'],
                )
                surr[s_idx] = compute_trialwise_mic(
                    shifted_epochs,
                    indices,
                    freqs,
                    MIC_CFG,
                    label=f'{subject} {event} {target_roi} {hemi} surr {s_idx}',
                )
            except Exception as exc:
                if verbose:
                    print(f'{subject} {event} {target_roi} {hemi}: surrogate {s_idx} failed:', exc)
                continue

        valid_surrogates = int(np.isfinite(surr).any(axis=1).sum())
        if valid_surrogates == 0:
            if verbose:
                print(f'{subject} {event} {target_roi} {hemi}: all surrogates failed')
            continue

        real_by_hemi[hemi] = real
        surr_by_hemi[hemi] = surr

    if len(real_by_hemi) == 0:
        return pd.DataFrame(), {'subject': subject, 'event': event, 'roi_target': target_roi, 'skip_reason': 'no_valid_hemispheres'}

    rows = []
    for _, trial in trial_map.iterrows():
        trial_idx = int(trial['trial_index'])
        hemi_rows = []
        for hemi in real_by_hemi:
            stats = scalar_null_stats(real_by_hemi[hemi][trial_idx], surr_by_hemi[hemi][:, trial_idx])
            if np.isfinite(stats['z']):
                hemi_rows.append(stats)
        if len(hemi_rows) == 0:
            continue

        rows.append({
            'participant': subject,
            'event': event,
            'trial_index': trial_idx,
            'trials_mem': trial.get('trials_mem', np.nan),
            'trials_gamble': trial.get('trials_gamble', np.nan),
            'rpe': trial.get('rpe', np.nan),
            'rpe_cat': trial.get('rpe_cat', np.nan),
            'memory_outcome': trial.get('memory_outcome', np.nan),
            'roi_source': MIC_CFG['roi_source'],
            'roi_target': target_roi,
            'mic_z': float(np.nanmean([h['z'] for h in hemi_rows])),
            'n_surrogates': int(np.round(np.nanmean([h['n'] for h in hemi_rows]))),
            'valid_hemi_count': int(len(hemi_rows)),
            'trial_col': trial_col,
        })

    out_df = pd.DataFrame(rows).sort_values('trial_index').reset_index(drop=True)
    meta = {
        'subject': subject,
        'event': event,
        'roi_target': target_roi,
        'skip_reason': '' if len(out_df) > 0 else 'no_successful_trials',
        'rows': int(len(out_df)),
        'valid_hemispheres': sorted(real_by_hemi.keys()),
        'n_epochs': int(len(epochs)),
        'n_aligned_trials': int(len(trial_map)),
        'n_surrogates_requested': int(n_surrogates),
        'n_cycles_effective': float(effective_n_cycles(epochs, MIC_CFG)),
    }
    return out_df, meta

def subject_cache_path(subject, event, target_roi, n_surrogates):
    key = {
        'subject': subject,
        'event': event,
        'target_roi': target_roi,
        'cfg': MIC_CFG,
        'n_surrogates': int(n_surrogates),
    }
    h = hashlib.sha1(_stable_json_dumps(key).encode('utf-8')).hexdigest()[:12]
    fname = f'{subject}_{event}_{MIC_CFG["roi_source"]}to{target_roi}_{h}.csv.gz'
    return os.path.join(subject_cache_dir, fname)

def run_or_load_subject_event_target(subject, event, target_roi, n_surrogates, overwrite=False, verbose=False):
    cache_file = subject_cache_path(subject, event, target_roi, n_surrogates)
    if (not overwrite) and os.path.exists(cache_file):
        df = pd.read_csv(cache_file)
        meta = {'subject': subject, 'event': event, 'roi_target': target_roi, 'cache_hit': True, 'cache_file': cache_file, 'rows': int(len(df)), 'skip_reason': ''}
        return df, meta

    df, meta = compute_subject_event_target_mic(subject, event, target_roi, n_surrogates, verbose=verbose)
    meta['cache_hit'] = False
    meta['cache_file'] = cache_file
    if len(df) > 0:
        os.makedirs(os.path.dirname(cache_file), exist_ok=True)
        df.to_csv(cache_file, index=False, compression='gzip')
    return df, meta


## Smoke Test

Runs one subject, one target ROI, and both events. This should finish before enabling the full batch.


In [ ]:
assert spectral_connectivity_time is not None, 'mne_connectivity.spectral_connectivity_time is required for this notebook'

smoke_df = pd.DataFrame()
smoke_log = []
if RUN['run_single_subject_smoke']:
    smoke_frames = []
    for event in MIC_CFG['events']:
        df_ev, meta_ev = run_or_load_subject_event_target(
            subject=RUN['smoke_subject'],
            event=event,
            target_roi=RUN['smoke_target_roi'],
            n_surrogates=RUN['n_surrogates_smoke'],
            overwrite=RUN['overwrite_subject_cache'],
            verbose=True,
        )
        smoke_log.append(meta_ev)
        if len(df_ev) > 0:
            smoke_frames.append(df_ev)
    smoke_df = pd.concat(smoke_frames, ignore_index=True) if len(smoke_frames) > 0 else pd.DataFrame()

print('Smoke rows:', len(smoke_df))
display(pd.DataFrame(smoke_log))
if len(smoke_df) > 0:
    required_cols = {'participant', 'event', 'trial_index', 'rpe_cat', 'roi_source', 'roi_target', 'mic_z', 'n_surrogates', 'valid_hemi_count'}
    assert required_cols.issubset(smoke_df.columns), f'Missing columns: {required_cols - set(smoke_df.columns)}'
    assert smoke_df['mic_z'].notna().all(), 'Smoke mic_z must be finite for successful rows'
    assert (smoke_df['n_surrogates'] > 0).all(), 'Smoke n_surrogates must be positive'
    assert (smoke_df['valid_hemi_count'] >= 1).all(), 'Smoke valid_hemi_count must be >= 1'
    display(smoke_df.head())


## Batch Run

Set `RUN['run_batch'] = True` above to compute all subjects, both events, and all target ROIs. Cached subject-level files are reused unless `overwrite_subject_cache=True`.


In [ ]:
RUN['run_batch'] = True

all_mic_df = pd.DataFrame()
cache_log_df = pd.DataFrame()

def run_batch_job(job):
    subject, event, target_roi = job
    try:
        df_job, meta_job = run_or_load_subject_event_target(
            subject=subject,
            event=event,
            target_roi=target_roi,
            n_surrogates=RUN['n_surrogates_batch'],
            overwrite=RUN['overwrite_subject_cache'],
            verbose=False,
        )
        meta_job = dict(meta_job)
        meta_job.setdefault('subject', subject)
        meta_job.setdefault('event', event)
        meta_job.setdefault('roi_target', target_roi)
        meta_job.setdefault('rows', int(len(df_job)))
        meta_job.setdefault('skip_reason', '')
        return df_job, meta_job
    except Exception as exc:
        return pd.DataFrame(), {
            'subject': subject,
            'event': event,
            'roi_target': target_roi,
            'rows': 0,
            'cache_hit': False,
            'cache_file': '',
            'skip_reason': f'job_failed:{type(exc).__name__}:{exc}',
        }

if RUN['run_batch']:
    jobs = [(s, ev, roi) for s in subjects for ev in MIC_CFG['events'] for roi in MIC_CFG['target_rois']]
    n_jobs_eff = max(1, int(RUN.get('n_jobs_batch', 1)))
    backend = RUN.get('parallel_backend', 'loky')
    print(f'Running {len(jobs)} MIC jobs with joblib n_jobs={n_jobs_eff}, backend={backend}')

    if n_jobs_eff == 1:
        results = [run_batch_job(job) for job in tqdm(jobs, desc='Trial-wise MIC batch')]
    else:
        results = Parallel(n_jobs=n_jobs_eff, backend=backend, verbose=10)(
            delayed(run_batch_job)(job) for job in jobs
        )

    frames = [df for df, _ in results if len(df) > 0]
    log_rows = [meta for _, meta in results]
    all_mic_df = pd.concat(frames, ignore_index=True) if len(frames) > 0 else pd.DataFrame()
    cache_log_df = pd.DataFrame(log_rows)
    cache_log_df.to_csv(analysis_paths['cache_log_csv'], index=False)
    if len(all_mic_df) > 0:
        all_mic_df = all_mic_df.sort_values(['participant', 'event', 'roi_target', 'trial_index']).reset_index(drop=True)
        all_mic_df.to_csv(analysis_paths['combined_csv'], index=False)
elif os.path.exists(analysis_paths['combined_csv']):
    all_mic_df = pd.read_csv(analysis_paths['combined_csv'])
    cache_log_df = pd.read_csv(analysis_paths['cache_log_csv']) if os.path.exists(analysis_paths['cache_log_csv']) else pd.DataFrame()
    print('Loaded existing combined cache:', analysis_paths['combined_csv'])
else:
    all_mic_df = smoke_df.copy()
    cache_log_df = pd.DataFrame(smoke_log)
    print('Batch disabled and no combined cache found; using smoke_df for downstream checks/plots.')

print('all_mic_df rows:', len(all_mic_df))
if len(cache_log_df) > 0:
    display(cache_log_df.head())
if len(all_mic_df) > 0:
    display(all_mic_df.head())


In [ ]:
all_mic_df = pd.DataFrame()
cache_log_df = pd.DataFrame()

if RUN['run_batch']:
    frames = []
    log_rows = []
    total_jobs = len(subjects) * len(MIC_CFG['events']) * len(MIC_CFG['target_rois'])
    iterator = tqdm(
        [(s, ev, roi) for s in subjects for ev in MIC_CFG['events'] for roi in MIC_CFG['target_rois']],
        total=total_jobs,
        desc='Trial-wise MIC batch',
    )
    for subject, event, target_roi in iterator:
        df_job, meta_job = run_or_load_subject_event_target(
            subject=subject,
            event=event,
            target_roi=target_roi,
            n_surrogates=RUN['n_surrogates_batch'],
            overwrite=RUN['overwrite_subject_cache'],
            verbose=False,
        )
        log_rows.append(meta_job)
        if len(df_job) > 0:
            frames.append(df_job)

    all_mic_df = pd.concat(frames, ignore_index=True) if len(frames) > 0 else pd.DataFrame()
    cache_log_df = pd.DataFrame(log_rows)
    cache_log_df.to_csv(analysis_paths['cache_log_csv'], index=False)
    if len(all_mic_df) > 0:
        all_mic_df.to_csv(analysis_paths['combined_csv'], index=False)
elif os.path.exists(analysis_paths['combined_csv']):
    all_mic_df = pd.read_csv(analysis_paths['combined_csv'])
    cache_log_df = pd.read_csv(analysis_paths['cache_log_csv']) if os.path.exists(analysis_paths['cache_log_csv']) else pd.DataFrame()
    print('Loaded existing combined cache:', analysis_paths['combined_csv'])
else:
    all_mic_df = smoke_df.copy()
    cache_log_df = pd.DataFrame(smoke_log)
    print('Batch disabled and no combined cache found; using smoke_df for downstream checks/plots.')

print('all_mic_df rows:', len(all_mic_df))
if len(cache_log_df) > 0:
    display(cache_log_df.head())
if len(all_mic_df) > 0:
    display(all_mic_df.head())


In [ ]:
# Batch / output integrity checks
if len(all_mic_df) == 0:
    print('No MIC rows available for checks.')
else:
    required_cols = {
        'participant', 'event', 'trial_index', 'trials_mem', 'trials_gamble', 'rpe', 'rpe_cat',
        'memory_outcome', 'roi_source', 'roi_target', 'mic_z', 'n_surrogates', 'valid_hemi_count'
    }
    assert required_cols.issubset(all_mic_df.columns), f'Missing required columns: {required_cols - set(all_mic_df.columns)}'
    assert not all_mic_df.duplicated(['participant', 'event', 'roi_target', 'trial_index']).any(), 'Rows must be unique per participant/event/target/trial'
    assert all_mic_df['roi_source'].eq(MIC_CFG['roi_source']).all(), 'Unexpected source ROI'
    assert set(all_mic_df['event'].dropna().unique()).issubset(set(MIC_CFG['events'].keys())), 'Unexpected event in output'
    assert all_mic_df['mic_z'].notna().all(), 'Successful rows must have finite mic_z'
    assert (all_mic_df['n_surrogates'] > 0).all(), 'n_surrogates must be positive'
    assert (all_mic_df['valid_hemi_count'] >= 1).all(), 'valid_hemi_count must be >= 1'
    if RUN['run_batch'] or os.path.exists(analysis_paths['combined_csv']):
        present_events = set(all_mic_df['event'].dropna().unique())
        assert set(MIC_CFG['events'].keys()).issubset(present_events), f'Missing events: {set(MIC_CFG["events"].keys()) - present_events}'
    print('Integrity checks passed.')


## Final Bar + Strip Plot

Subject-level points average `mic_z` across trials within each event and target ROI.


In [ ]:
if len(all_mic_df) == 0:
    print('No MIC rows available for final plot.')
else:
    summary_df = (
        all_mic_df
        .groupby(['participant', 'event', 'roi_source', 'roi_target'], as_index=False)
        .agg(
            mic_z=('mic_z', 'mean'),
            n_trials=('trial_index', 'nunique'),
            n_surrogates=('n_surrogates', 'mean'),
            valid_hemi_count=('valid_hemi_count', 'mean'),
        )
    )
    summary_df.to_csv(analysis_paths['summary_csv'], index=False)

    target_order = [roi for roi in MIC_CFG['target_rois'] if roi in summary_df['roi_target'].dropna().unique()]
    event_order = [ev for ev in MIC_CFG['events'].keys() if ev in summary_df['event'].dropna().unique()]
    event_order = ['cue_on']
    assert len(target_order) > 0, 'No target ROIs available for plotting'
    assert len(event_order) > 0, 'No events available for plotting'

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    palette = {'feedback_start': '#c75146', 'cue_on': '#2f6f9f'}
    sns.barplot(
        data=summary_df,
        x='roi_target',
        y='mic_z',
        hue='event',
        order=target_order,
        hue_order=event_order,
        errorbar='se',
        palette=palette,
        alpha=0.78,
        ax=ax,
    )
    # sns.stripplot(
    #     data=summary_df,
    #     x='roi_target',
    #     y='mic_z',
    #     hue='event',
    #     order=target_order,
    #     hue_order=event_order,
    #     dodge=True,
    #     palette=palette,
    #     alpha=0.65,
    #     edgecolor='black',
    #     linewidth=0.3,
    #     size=3,
    #     ax=ax,
    #     zorder=10,
    # )
    handles, labels = ax.get_legend_handles_labels()
    if len(handles) > len(event_order):
        handles = handles[:len(event_order)]
        labels = labels[:len(event_order)]
    ax.legend(handles, labels, title='Event', loc='upper right')
    ax.axhline(0, color='gray', linewidth=1)
    ax.set_xlabel('Target ROI')
    ax.set_ylabel('Trial-wise MIC z-score')
    ax.set_title(f"{MIC_CFG['roi_source']} -> targets | theta {MIC_CFG['fmin']}-{MIC_CFG['fmax']} Hz")
    sns.despine()
    plt.tight_layout()
    fig.savefig(analysis_paths['bar_strip_pdf'], dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(analysis_paths['bar_strip_png'], dpi=300, bbox_inches='tight', facecolor='white')
    print('Saved:', analysis_paths['bar_strip_pdf'])
    display(summary_df.head())
    plt.show()


## Optional Posthoc RPE Split

This cell demonstrates posthoc splitting by `rpe_cat` from the cached trial-wise table. It does not recompute MIC.


In [ ]:
# Optional: posthoc neutral vs positive RPE split. Edit rpe_keep for other contrasts.
rpe_keep = ['neg', 'neutral', 'pos']

if len(all_mic_df) == 0:
    print('No MIC rows available for RPE split plot.')
elif 'rpe_cat' not in all_mic_df.columns:
    print('No rpe_cat column found.')
else:
    rpe_plot_df = all_mic_df.copy()
    rpe_plot_df['rpe_cat'] = normalize_rpe_cat(rpe_plot_df['rpe_cat'])
    rpe_plot_df = rpe_plot_df[rpe_plot_df['rpe_cat'].isin(rpe_keep)].copy()
    if len(rpe_plot_df) == 0:
        print('No rows after filtering to rpe_keep =', rpe_keep)
    else:
        rpe_summary_df = (
            rpe_plot_df
            .groupby(['participant', 'event', 'roi_target', 'rpe_cat'], as_index=False)
            .agg(mic_z=('mic_z', 'mean'), n_trials=('trial_index', 'nunique'))
        )
        target_order = [roi for roi in MIC_CFG['target_rois'] if roi in rpe_summary_df['roi_target'].dropna().unique()]
        event_order = [ev for ev in MIC_CFG['events'].keys() if ev in rpe_summary_df['event'].dropna().unique()]
        palette = {'neutral': '#7f7f7f', 'pos': '#2ca02c', 'neg': '#d62728'}

        fig, axes = plt.subplots(1, len(event_order), figsize=(5 * len(event_order), 4.5), sharey=True)
        if len(event_order) == 1:
            axes = [axes]
        for ax, event in zip(axes, event_order):
            ev_df = rpe_summary_df[rpe_summary_df['event'].eq(event)].copy()
            sns.barplot(
                data=ev_df,
                x='roi_target',
                y='mic_z',
                hue='rpe_cat',
                order=target_order,
                hue_order=rpe_keep,
                errorbar='se',
                palette=palette,
                alpha=0.78,
                ax=ax,
            )
            # sns.stripplot(
            #     data=ev_df,
            #     x='roi_target',
            #     y='mic_z',
            #     hue='rpe_cat',
            #     order=target_order,
            #     hue_order=rpe_keep,
            #     dodge=True,
            #     palette=palette,
            #     alpha=0.65,
            #     edgecolor='black',
            #     linewidth=0.3,
            #     size=3,
            #     ax=ax,
            #     zorder=10,
            # )
            handles, labels = ax.get_legend_handles_labels()
            if len(handles) > len(rpe_keep):
                handles = handles[:len(rpe_keep)]
                labels = labels[:len(rpe_keep)]
            ax.legend(handles, labels, title='RPE', loc='upper right')
            ax.axhline(0, color='gray', linewidth=1)
            ax.set_title(event)
            ax.set_xlabel('Target ROI')
            ax.set_ylabel('Trial-wise MIC z-score')
        sns.despine()
        plt.tight_layout()
        fig.savefig(analysis_paths['rpe_split_pdf'], dpi=300, bbox_inches='tight', facecolor='white')
        fig.savefig(analysis_paths['rpe_split_png'], dpi=300, bbox_inches='tight', facecolor='white')
        print('Saved:', analysis_paths['rpe_split_pdf'])
        display(rpe_summary_df.head())
        plt.show()
